# Attention Mechanism

> Based on Stanford CS230 — C5M3, Andrew Ng / DeepLearning.AI  
> Original paper: Bahdanau et al. (2015)

The fixed-size context vector of seq2seq is a **bottleneck**: the entire source sentence must be compressed into one vector. **Attention** lets the decoder look at all encoder hidden states and dynamically decide which parts of the input to focus on at each output step.

## Learning Objectives

1. Explain the information bottleneck problem in vanilla seq2seq
2. Compute attention weights $\alpha^{\langle t, t' \rangle}$
3. Form the context vector as a weighted sum over encoder states
4. Visualise an attention matrix and interpret it

## Attention Equations

For each decoder step $t$, compute an **attention score** $e^{\langle t, t' \rangle}$ for every encoder step $t'$:

$$e^{\langle t, t' \rangle} = a\!\bigl(s^{\langle t-1 \rangle},\, h^{\langle t' \rangle}\bigr)$$

Normalise with softmax to get **attention weights**:

$$\alpha^{\langle t, t' \rangle} = \frac{\exp(e^{\langle t, t' \rangle})}{\sum_{t''} \exp(e^{\langle t, t'' \rangle})}$$

Form the **context vector** for step $t$:

$$c^{\langle t \rangle} = \sum_{t'} \alpha^{\langle t, t' \rangle}\, h^{\langle t' \rangle}$$

- $s^{\langle t-1 \rangle}$ — previous decoder hidden state
- $h^{\langle t' \rangle}$ — encoder hidden state at position $t'$
- $\sum_{t'} \alpha^{\langle t, t' \rangle} = 1$ for each decoder step $t$

## Attention Matrix Interpretation

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 560 200" width="560" height="200" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Encoder labels (x-axis) -->
  <text x="100" y="15" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">Je</text>
  <text x="185" y="15" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">suis</text>
  <text x="270" y="15" text-anchor="middle" fill="#b03a3a" font-size="10" font-weight="bold">étudiant</text>
  <!-- Decoder labels (y-axis) -->
  <text x="68"  y="55"  text-anchor="end" fill="#8a3ab8" font-size="10" font-weight="bold">I</text>
  <text x="68"  y="110" text-anchor="end" fill="#8a3ab8" font-size="10" font-weight="bold">am</text>
  <text x="68"  y="165" text-anchor="end" fill="#8a3ab8" font-size="10" font-weight="bold">student</text>
  <!-- Attention cells: row=decoder, col=encoder -->
  <!-- Row 1: I → Je(high), suis(low), étudiant(low) -->
  <rect x="75"  y="25" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.90"/>
  <rect x="160" y="25" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.08"/>
  <rect x="245" y="25" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.05"/>
  <text x="100" y="49" text-anchor="middle" fill="white"  font-size="11" font-weight="bold">0.92</text>
  <text x="185" y="49" text-anchor="middle" fill="#3a7bbf" font-size="11">0.05</text>
  <text x="270" y="49" text-anchor="middle" fill="#3a7bbf" font-size="11">0.03</text>
  <!-- Row 2: am → Je(low), suis(high), étudiant(low) -->
  <rect x="75"  y="80" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.10"/>
  <rect x="160" y="80" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.88"/>
  <rect x="245" y="80" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.07"/>
  <text x="100"  y="104" text-anchor="middle" fill="#3a7bbf" font-size="11">0.08</text>
  <text x="185"  y="104" text-anchor="middle" fill="white"  font-size="11" font-weight="bold">0.88</text>
  <text x="270"  y="104" text-anchor="middle" fill="#3a7bbf" font-size="11">0.04</text>
  <!-- Row 3: student → Je(low), suis(low), étudiant(high) -->
  <rect x="75"  y="135" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.05"/>
  <rect x="160" y="135" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.10"/>
  <rect x="245" y="135" width="50" height="40" rx="3" fill="#5b9bd5" fill-opacity="0.90"/>
  <text x="100"  y="159" text-anchor="middle" fill="#3a7bbf" font-size="11">0.03</text>
  <text x="185"  y="159" text-anchor="middle" fill="#3a7bbf" font-size="11">0.08</text>
  <text x="270"  y="159" text-anchor="middle" fill="white"  font-size="11" font-weight="bold">0.89</text>
  <!-- Axis labels -->
  <text x="185"  y="192" text-anchor="middle" fill="#888" font-size="9">Encoder input (French)</text>
  <text x="25"   y="110" text-anchor="middle" fill="#888" font-size="9" transform="rotate(-90,25,110)">Decoder output (English)</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

# ---- Bahdanau-style additive attention ----
class Attention:
    """Additive (Bahdanau) attention: score = v^T tanh(W1*s + W2*h)"""
    def __init__(self, n_a, n_d, n_att=8, seed=0):
        np.random.seed(seed)
        s = 0.05
        self.W1 = np.random.randn(n_att, n_d) * s   # decoder state
        self.W2 = np.random.randn(n_att, n_a) * s   # encoder state
        self.v  = np.random.randn(n_att) * s

    def score(self, s_prev, h_t):
        """s_prev: (n_d,), h_t: (n_a,)"""
        e = self.v @ np.tanh(self.W1 @ s_prev + self.W2 @ h_t)
        return e

    def context(self, s_prev, H):
        """H: (T_x, n_a), returns context (n_a,) and alphas (T_x,)"""
        scores = np.array([self.score(s_prev, H[t]) for t in range(len(H))])
        alphas = softmax(scores)
        ctx    = alphas @ H
        return ctx, alphas

# ---- Simulate attention over a translation task ----
np.random.seed(42)
T_x, T_y = 6, 5
n_a, n_d, n_att = 16, 16, 10

H = np.random.randn(T_x, n_a)  # encoder hidden states
attn = Attention(n_a, n_d, n_att)

attention_matrix = []
decoder_state = np.zeros(n_d)
for t in range(T_y):
    ctx, alphas = attn.context(decoder_state, H)
    attention_matrix.append(alphas)
    # update decoder state (simplified: just add context projection)
    decoder_state = np.tanh(ctx[:n_d])

attention_matrix = np.array(attention_matrix)   # (T_y, T_x)

src_words = ['The', 'cat', 'sat', 'on', 'the', 'mat']
tgt_words = ['Le', 'chat', 'était', 'sur', 'le']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Attention Mechanism', fontsize=13, fontweight='bold')

# Attention heatmap
ax = axes[0]
im = ax.imshow(attention_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=attention_matrix.max())
ax.set_xticks(range(T_x)); ax.set_xticklabels(src_words, rotation=45, ha='right')
ax.set_yticks(range(T_y)); ax.set_yticklabels(tgt_words)
ax.set_xlabel('Encoder position (source)')
ax.set_ylabel('Decoder position (target)')
ax.set_title('Attention weight matrix $\\alpha^{\\langle t, t\\'\\rangle}$')
for i in range(T_y):
    for j in range(T_x):
        ax.text(j, i, f'{attention_matrix[i,j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if attention_matrix[i,j] > 0.25 else '#333')
plt.colorbar(im, ax=ax, label='attention weight')

# Per-decoder-step attention distribution
ax = axes[1]
for t in range(T_y):
    ax.plot(range(T_x), attention_matrix[t], 'o-', color=P[t % len(P)], lw=1.8,
            label=f'decoder step {t+1} ({tgt_words[t]})', ms=6)
ax.set_xticks(range(T_x))
ax.set_xticklabels(src_words, rotation=30, ha='right')
ax.set_xlabel('Encoder position')
ax.set_ylabel('Attention weight')
ax.set_title('Attention Distribution per Decoder Step')
ax.legend(fontsize=8)
ax.grid(True)

plt.tight_layout()
plt.show()
